In [1]:
%%capture
%pip install -U bitsandbytes
%pip install -U transformers
%pip install -U accelerate
%pip install -U peft
%pip install -U trl

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
login(token = hf_token)

In [3]:
base_model = "meta-llama/Llama-3.1-8B-Instruct"
new_model = "/kaggle/input/notebooks/pranavhari1345/fine-tune-llama-3-1-8b-on-medical-dataset/llama-3.1-8b-chat-doctor/"

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from peft import PeftModel
import torch

# Reload tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(base_model)
torch_dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model_reload = AutoModelForCausalLM.from_pretrained(
        base_model,
        return_dict=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map = "auto",
        quantization_config=bnb_config,
)

# Merge adapter with base model
model = PeftModel.from_pretrained(base_model_reload, new_model)

model = model.merge_and_unload()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [5]:
messages = [{"role": "user", "content": "Hello doctor, I have bad acne. How do I get rid of it?"}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device="cuda",
)

outputs = pipe(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello doctor, I have bad acne. How do I get rid of it?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hello! I'm not a doctor, but I can provide some general information that may help. To get rid of acne, it's essential to understand that it's a complex condition caused by a combination of factors, including genetics, hormones, and bacteria. Here are some steps you can take:

1.  **Maintain good hygiene**: Wash your face twice a day with a gentle cleanser to remove dirt, oil, and bacteria. Avoid over-washing, which can strip your skin of its natural oils and make acne worse.
2.  **Use non-comedogenic products**: Choose products


In [6]:
model.save_pretrained("llama-3.1-8b-chat-doctor")
tokenizer.save_pretrained("llama-3.1-8b-chat-doctor")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('llama-3.1-8b-chat-doctor/tokenizer_config.json',
 'llama-3.1-8b-chat-doctor/chat_template.jinja',
 'llama-3.1-8b-chat-doctor/tokenizer.json')

In [7]:
model.push_to_hub("llama-3.1-8b-chat-doctor")
tokenizer.push_to_hub("llama-3.1-8b-chat-doctor")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/pranavhari12/llama-3.1-8b-chat-doctor/commit/dbb08a119d309b1565a160bcbc9fc89694cee0be', commit_message='Upload tokenizer', commit_description='', oid='dbb08a119d309b1565a160bcbc9fc89694cee0be', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pranavhari12/llama-3.1-8b-chat-doctor', endpoint='https://huggingface.co', repo_type='model', repo_id='pranavhari12/llama-3.1-8b-chat-doctor'), pr_revision=None, pr_num=None)